# MRI preprocessing

Run cells in order after editing the paths below. DICOM conversion is optional if NIfTI images already exist.

Python dependencies: pandas, numpy, nibabel, scipy, tqdm. Install dcm2niix and HD-BET separately and make their executables available on PATH. Use the HD-BET version and command-line options from your study environment.

**Methodology review needed:** the supplied order is DICOM conversion → intensity normalization and resizing → HD-BET skull stripping. This differs from skull stripping before normalization and resizing. The original normalization function clips the entire array using percentiles from positive voxels; this changes background zeros and is not brain-mask-only normalization. It does not explicitly replace NaNs or cap the 99.9th percentile.


## Paths and imports

In [ ]:
from pathlib import Path
import shutil
import subprocess
import pandas as pd
import nibabel as nib
import numpy as np
from scipy.ndimage import zoom
from tqdm.auto import tqdm

# Edit these paths for your environment.
RAW_DICOM_DIR = Path("data/T1w_DICOM")
NIFTI_DIR = Path("data/T1w_nifti")
INPUT_CSV = Path("data/OASIS3.csv")
PREPROCESSED_DIR = Path("data/T1w_preprocessed_224")
PREPROCESSED_CSV = Path("data/T1w_nifti_preprocessed.csv")
SKULLSTRIP_DIR = Path("data/T1w_preprocessed_skullstrip")
SKULLSTRIP_CSV = Path("data/T1w_nifti_skullstrip.csv")
TARGET_SHAPE = (224, 224, 224)
HD_BET_DEVICE = "cuda"


## DICOM to NIfTI (optional)

This discovers folders containing `.dcm` or `.DCM` files. The input CSV must point to the resulting NIfTI files; conversion does not build that mapping automatically. Check conversion filenames for collisions across sessions.

In [ ]:
NIFTI_DIR.mkdir(parents=True, exist_ok=True)
if shutil.which("dcm2niix") is None:
    raise FileNotFoundError("dcm2niix is not available on PATH.")

#A DICOM series folder is usually the folder containing many .dcm files
dcm_files = list(RAW_DICOM_DIR.rglob("*.dcm")) + list(RAW_DICOM_DIR.rglob("*.DCM"))

series_dirs = sorted(set([p.parent for p in dcm_files]))

print("DICOM files:", len(dcm_files))
print("Series folders:", len(series_dirs))

for i, series_dir in enumerate(series_dirs, start=1):
     print(f"[{i}/{len(series_dirs)}] Converting:", series_dir)

     cmd = [
         "dcm2niix",
         "-z", "y",
         "-o", str(NIFTI_DIR),
         "-f", "%p_%s_%t",
         str(series_dir)
     ]

     subprocess.run(cmd, check=True)

## Intensity normalization and resizing

Input CSV columns: `MRI_session_label` (unique scan/session identifier) and `local_image_path` (path to the input `.nii` or `.nii.gz` MRI). Inputs must be 3D T1-weighted volumes.

In [ ]:
def normalize_intensity(volume):
     volume = volume.astype(np.float32)

     nonzero = volume[volume > 0]

     if len(nonzero) == 0:
         return volume

     p1, p99 = np.percentile(nonzero, [1, 99])
     volume = np.clip(volume, p1, p99)

     nonzero = volume[volume > 0]
     mean = nonzero.mean()
     std = nonzero.std()

     if std > 0:
         volume = (volume - mean) / std

     return volume


def resize_volume(volume, target_shape=(224, 224, 224)):
     resize_factors = [
         target_shape[i] / volume.shape[i]
         for i in range(3)
     ]

     volume_resized = zoom(volume, resize_factors, order=1)

     return volume_resized


def preprocess_mri(input_path, output_path, target_shape=(224, 224, 224)):
     img = nib.load(input_path)
     volume = img.get_fdata()

     volume = normalize_intensity(volume)
     volume = resize_volume(volume, target_shape)

     volume = volume.astype(np.float32)

     new_img = nib.Nifti1Image(volume, affine=np.eye(4))
     nib.save(new_img, output_path)



In [ ]:
df = pd.read_csv(INPUT_CSV)

required = ["MRI_session_label", "local_image_path"]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing CSV columns: {missing}")
if df[required].isna().any().any():
    raise ValueError("Image paths and session labels must not be missing.")
if df["MRI_session_label"].duplicated().any():
    raise ValueError("MRI_session_label must be unique to prevent output overwriting.")
    
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)

preprocessed_paths = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    input_path = Path(row["local_image_path"])
    output_name = f"{row['MRI_session_label']}_T1w_224.nii.gz"
    output_path = PREPROCESSED_DIR / output_name
    if not output_path.exists():
        preprocess_mri(str(input_path), str(output_path), TARGET_SHAPE)
    preprocessed_paths.append(str(output_path))
    
df["preprocessed_image_path"] = preprocessed_paths
df.to_csv(PREPROCESSED_CSV, index=False)
print("Saved:", PREPROCESSED_CSV)


## Skull stripping using HD-BET

Reads the `preprocessed_image_path` column generated above. The output CSV records `image_path_hdbet` and `hdbet_status`.

In [ ]:
df = pd.read_csv(PREPROCESSED_CSV)
input_col = "preprocessed_image_path"
SKULLSTRIP_DIR.mkdir(parents=True, exist_ok=True)
SKULLSTRIP_CSV.parent.mkdir(parents=True, exist_ok=True)
df["image_path_hdbet"] = None
df["hdbet_status"] = None
df.to_csv(SKULLSTRIP_CSV, index=False)


In [ ]:
hdbet_exe = shutil.which("hd-bet")
if hdbet_exe is None:
    raise FileNotFoundError("hd-bet is not available on PATH.")

for idx, row in tqdm(df.iterrows(), total=len(df)):

    input_path = Path(row[input_col])

    output_name = input_path.name.replace(".nii.gz", "_brain.nii.gz")
    output_path = SKULLSTRIP_DIR / output_name

    # Skip already successful cases
    if df.loc[idx, "hdbet_status"] in ["success", "already_exists"]:
        continue

    # If output already exists, save path and status immediately
    if output_path.exists():
        df.loc[idx, "image_path_hdbet"] = str(output_path)
        df.loc[idx, "hdbet_status"] = "already_exists"

        # Save dataframe immediately
        df.to_csv(SKULLSTRIP_CSV, index=False)
        continue

    cmd = [
        hdbet_exe,
        "-i", str(input_path),
        "-o", str(output_path),
        "-device", HD_BET_DEVICE
    ]

    try:
        subprocess.run(cmd, check=True)
        if not output_path.is_file():
            raise FileNotFoundError(f"HD-BET did not produce {output_path}")

        df.loc[idx, "image_path_hdbet"] = str(output_path)
        df.loc[idx, "hdbet_status"] = "success"

    except (subprocess.CalledProcessError, FileNotFoundError):
        print("Failed:", input_path)

        df.loc[idx, "image_path_hdbet"] = None
        df.loc[idx, "hdbet_status"] = "failed"

    # Save dataframe after every image
    df.to_csv(SKULLSTRIP_CSV, index=False)

print("Done.")
print(df["hdbet_status"].value_counts(dropna=False))
print("Saved updated CSV:", SKULLSTRIP_CSV)